# BLDC Motor Fault Detection — Psychoacoustic Features

Binary classification (**Healthy** vs **Faulty**) of a BLDC motor from four psychoacoustic
features — **Loudness, Sharpness, Fluctuation, Roughness** — with four models:

| Model | Library | Training loss |
|---|---|---|
| Logistic Regression | scikit-learn `SGDClassifier(loss="log_loss")` | log loss |
| SVM (RBF kernel) | scikit-learn `Nystroem` + `SGDClassifier(loss="hinge")` | hinge loss |
| ANN (MLP) | PyTorch | binary cross-entropy |
| 1D-CNN | PyTorch | binary cross-entropy |

**Why no pretrained model?** Each input is just 4 numbers computed from one specific motor
recording setup. Public pretrained audio models (e.g. PANNs, AST, wav2vec) take raw
waveforms or spectrograms, not these 4 features, so there is nothing to fine-tune. The models
here are tiny and train in about a minute even on a CPU runtime (a GPU is used if available).

### How to run on Colab
1. *Runtime → Run all*.
2. When the upload widget appears, select the four CSV files
   (`Good_pure.csv`, `Broken_pure.csv`, `Noisy_Good.csv`, `Noisy_Broken.csv`)
   **or** a single `.zip` of the `data/` folder.
3. Plots are shown inline; `outputs/` (PNG plots + `metrics.json`) is offered as a download at the end.

In [ ]:
# Colab already ships numpy, pandas, scikit-learn, torch, matplotlib and seaborn.
# Running locally instead?  uv sync && uv run jupyter lab   (or: pip install -r the deps)
import copy
import io
import json
import zipfile
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from numpy.typing import NDArray
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    hinge_loss,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
FEATURES = ["Loudness", "Sharpness", "Fluctuation", "Roughness"]
CLASS_NAMES = ["Healthy", "Faulty"]

SKLEARN_EPOCHS = 100
TORCH_EPOCHS = 150
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

FloatArray = NDArray[np.float32]
IntArray = NDArray[np.int64]

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__)

## 1. Get the data

In [ ]:
try:
    from google.colab import files  # only exists on Colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not any(DATA_DIR.rglob("*.csv")):
    if not IN_COLAB:
        raise FileNotFoundError(f"Put the psychoacoustic CSVs under {DATA_DIR.resolve()}")
    DATA_DIR.mkdir(exist_ok=True)
    print("Select the 4 psychoacoustic CSV files (or one .zip containing them)")
    for name, content in files.upload().items():
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        else:
            (DATA_DIR / name).write_bytes(content)

print(*sorted(str(p) for p in DATA_DIR.rglob("*.csv")), sep="\n")

In [ ]:
def load_dataset(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Load every CSV and attach `label` (0=healthy, 1=faulty) and `noise` condition."""
    frames: list[pd.DataFrame] = []
    for csv in sorted(data_dir.rglob("*.csv")):
        if csv.name.startswith("._"):  # macOS zip artefacts
            continue
        df = pd.read_csv(csv)
        df["label"] = 1 if "broken" in csv.stem.lower() else 0
        df["noise"] = "noisy" if "noisy" in csv.stem.lower() else "pure"
        df["source"] = csv.name
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No CSV files found under {data_dir}")
    return pd.concat(frames, ignore_index=True)


df = load_dataset()
print(f"{len(df)} samples")
display(df.groupby(["source", "label"]).size().rename("rows").to_frame())
display(df.groupby("source")[FEATURES].mean())

### Quick look at the features
`Window` is only a running index of the analysis window, so it is **not** used as a feature.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), layout="constrained")
plot_df = df.assign(state=df["label"].map(dict(enumerate(CLASS_NAMES))))
for ax, feat in zip(axes, FEATURES, strict=True):
    sns.boxplot(data=plot_df, x="noise", y=feat, hue="state", ax=ax, palette=["#2563eb", "#dc2626"])
    ax.set_title(feat)
plt.show()

## 2. Train / validation / test split
Stratified **70 / 15 / 15** split (stratified on label *and* noise condition).
`StandardScaler` is fit on the training set only — `Fluctuation` is ~1e-6 while
`Loudness` is ~40, so scaling matters for every model here.

In [ ]:
@dataclass
class Split:
    x_train: FloatArray
    y_train: IntArray
    x_val: FloatArray
    y_val: IntArray
    x_test: FloatArray
    y_test: IntArray


def make_split(df: pd.DataFrame) -> Split:
    x = df[FEATURES].to_numpy(dtype=np.float32)
    y = df["label"].to_numpy(dtype=np.int64)
    strata = df["label"].astype(str) + "_" + df["noise"]
    x_train, x_tmp, y_train, y_tmp, _, s_tmp = train_test_split(
        x, y, strata, test_size=0.30, stratify=strata, random_state=SEED
    )
    x_val, x_test, y_val, y_test = train_test_split(
        x_tmp, y_tmp, test_size=0.50, stratify=s_tmp, random_state=SEED
    )
    scaler = StandardScaler().fit(x_train)
    return Split(
        x_train=scaler.transform(x_train).astype(np.float32),
        y_train=y_train,
        x_val=scaler.transform(x_val).astype(np.float32),
        y_val=y_val,
        x_test=scaler.transform(x_test).astype(np.float32),
        y_test=y_test,
    )


split = make_split(df)
print(f"train={len(split.y_train)}  val={len(split.y_val)}  test={len(split.y_test)}")

## 3. Models

### 3a. Logistic Regression & SVM (scikit-learn)
`LogisticRegression` / `SVC` do not expose a per-iteration loss, so both models are trained
with `SGDClassifier.partial_fit`, one epoch at a time, which gives the loss curves:

* **Logistic Regression** → `loss="log_loss"` (exactly logistic regression, fit by SGD).
* **SVM** → RBF kernel approximated with `Nystroem` (300 components), then `loss="hinge"`
  (a linear SVM in that kernel feature space, i.e. an approximate RBF-SVM).

In [ ]:
@dataclass
class History:
    train: list[float] = field(default_factory=list)
    val: list[float] = field(default_factory=list)


@dataclass
class ModelResult:
    name: str
    loss_name: str
    history: History
    y_pred: IntArray


LossFn = Callable[[SGDClassifier, FloatArray, IntArray], float]


def _log_loss(clf: SGDClassifier, x: FloatArray, y: IntArray) -> float:
    return float(log_loss(y, clf.predict_proba(x), labels=[0, 1]))


def _hinge_loss(clf: SGDClassifier, x: FloatArray, y: IntArray) -> float:
    return float(hinge_loss(y, clf.decision_function(x), labels=[0, 1]))


def train_sgd(
    clf: SGDClassifier,
    split: Split,
    loss_fn: LossFn,
    *,
    x_train: FloatArray,
    x_val: FloatArray,
    epochs: int = SKLEARN_EPOCHS,
) -> History:
    rng = np.random.default_rng(SEED)
    history = History()
    classes = np.array([0, 1])
    for _ in range(epochs):
        order = rng.permutation(len(x_train))
        clf.partial_fit(x_train[order], split.y_train[order], classes=classes)
        history.train.append(loss_fn(clf, x_train, split.y_train))
        history.val.append(loss_fn(clf, x_val, split.y_val))
    return history


def run_logistic_regression(split: Split) -> ModelResult:
    clf = SGDClassifier(
        loss="log_loss", alpha=1e-4, learning_rate="constant", eta0=0.01, random_state=SEED
    )
    history = train_sgd(clf, split, _log_loss, x_train=split.x_train, x_val=split.x_val)
    return ModelResult("Logistic Regression", "log loss", history, clf.predict(split.x_test))


def run_svm(split: Split) -> ModelResult:
    kernel = Nystroem(kernel="rbf", gamma=0.5, n_components=300, random_state=SEED)
    x_train = kernel.fit_transform(split.x_train).astype(np.float32)
    x_val = kernel.transform(split.x_val).astype(np.float32)
    x_test = kernel.transform(split.x_test).astype(np.float32)
    clf = SGDClassifier(
        loss="hinge", alpha=1e-4, learning_rate="constant", eta0=0.005, random_state=SEED
    )
    history = train_sgd(clf, split, _hinge_loss, x_train=x_train, x_val=x_val)
    return ModelResult("SVM (RBF kernel)", "hinge loss", history, clf.predict(x_test))

### 3b. ANN & 1D-CNN (PyTorch)
* **ANN**: 4 → 32 → 16 → 1 MLP with ReLU and dropout.
* **1D-CNN**: the 4-feature vector is treated as a 1-channel signal of length 4:
  two `Conv1d(k=3)` + BatchNorm + ReLU blocks → flatten → dense head.

Both use `BCEWithLogitsLoss` with Adam (lr 1e-3, batch 64, 150 epochs). The weights with the
lowest validation loss are restored before testing.

In [ ]:
class ANN(nn.Module):
    def __init__(self, n_features: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


class CNN1D(nn.Module):
    def __init__(self, n_features: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * n_features, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)  # (batch, features) -> (batch, 1 channel, features)
        return self.classifier(self.features(x)).squeeze(-1)


@torch.no_grad()
def _eval_loss(model: nn.Module, x: torch.Tensor, y: torch.Tensor, criterion: nn.Module) -> float:
    model.eval()
    return float(criterion(model(x), y).item())


@torch.no_grad()
def _predict(model: nn.Module, x: torch.Tensor) -> IntArray:
    model.eval()
    return (torch.sigmoid(model(x)) > 0.5).long().cpu().numpy()


def train_torch(
    model: nn.Module, split: Split, device: torch.device, epochs: int = TORCH_EPOCHS
) -> tuple[History, IntArray]:
    model.to(device)
    x_train = torch.from_numpy(split.x_train).to(device)
    y_train = torch.from_numpy(split.y_train).float().to(device)
    x_val = torch.from_numpy(split.x_val).to(device)
    y_val = torch.from_numpy(split.y_val).float().to(device)
    x_test = torch.from_numpy(split.x_test).to(device)

    loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

    history = History()
    best_val = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        history.train.append(_eval_loss(model, x_train, y_train, criterion))
        history.val.append(_eval_loss(model, x_val, y_val, criterion))
        if history.val[-1] < best_val:
            best_val = history.val[-1]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return history, _predict(model, x_test)


def run_ann(split: Split, device: torch.device) -> ModelResult:
    torch.manual_seed(SEED)
    history, y_pred = train_torch(ANN(len(FEATURES)), split, device)
    return ModelResult("ANN (MLP)", "BCE loss", history, y_pred)


def run_cnn(split: Split, device: torch.device) -> ModelResult:
    torch.manual_seed(SEED)
    history, y_pred = train_torch(CNN1D(len(FEATURES)), split, device)
    return ModelResult("1D-CNN", "BCE loss", history, y_pred)

## 4. Train all four models

In [ ]:
results: list[ModelResult] = []
for runner in (run_logistic_regression, run_svm):
    results.append(runner(split))
    print("trained", results[-1].name)
for runner in (run_ann, run_cnn):
    results.append(runner(split, device))
    print("trained", results[-1].name)

## 5. Loss curves

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
for ax, res in zip(axes.flatten(), results, strict=True):
    epochs = np.arange(1, len(res.history.train) + 1)
    ax.plot(epochs, res.history.train, label="train", color="#2563eb", linewidth=2)
    ax.plot(epochs, res.history.val, label="validation", color="#ea580c", linewidth=2)
    ax.set_title(res.name)
    ax.set_xlabel("epoch")
    ax.set_ylabel(res.loss_name)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle("Training loss per epoch", fontsize=14)
fig.savefig(OUT_DIR / "loss_curves.png", dpi=150)
plt.show()

## 6. Confusion matrices & metrics (held-out test set)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9), layout="constrained")
for ax, res in zip(axes.flatten(), results, strict=True):
    cm = confusion_matrix(split.y_test, res.y_pred, labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
        annot_kws={"size": 14},
    )
    ax.set_title(f"{res.name}  (accuracy {accuracy_score(split.y_test, res.y_pred):.2%})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
fig.suptitle("Confusion matrices on the held-out test set", fontsize=14)
fig.savefig(OUT_DIR / "confusion_matrices.png", dpi=150)
plt.show()

metrics = {
    res.name: {
        "accuracy": float(accuracy_score(split.y_test, res.y_pred)),
        "precision": float(precision_score(split.y_test, res.y_pred, zero_division=0)),
        "recall": float(recall_score(split.y_test, res.y_pred, zero_division=0)),
        "f1": float(f1_score(split.y_test, res.y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(split.y_test, res.y_pred, labels=[0, 1]).tolist(),
    }
    for res in results
}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
display(pd.DataFrame(metrics).T.drop(columns="confusion_matrix").astype(float).round(4))

## 7. Download the outputs (Colab)

In [ ]:
if IN_COLAB:
    import shutil

    shutil.make_archive("outputs", "zip", OUT_DIR)
    files.download("outputs.zip")